In [0]:
%pip install openai sentence-transformers "chromadb==0.5.23" "numpy<2.0" langchain langchain-openai
dbutils.library.restartPython()

In [0]:
import os
import re
import hashlib
import shutil
from pathlib import Path
from typing import List, Dict, Optional
from sentence_transformers import SentenceTransformer
import chromadb
from openai import OpenAI

print("Imports OK")

In [0]:
SKIP_SECTIONS = {"links", "contact", "languages", "interests"}

def chunk_markdown(text: str, source: str) -> List[Dict]:
    chunks = []
    sections = re.split(r'\n(?=## )', text.strip())
    for section in sections:
        if not section.strip():
            continue
        lines = section.strip().split('\n')
        heading = lines[0].replace('#', '').strip() if lines[0].startswith('#') else "intro"
        content = '\n'.join(lines).strip()
        if len(content) < 150:
            continue
        if heading.lower() in SKIP_SECTIONS:
            continue
        chunk_id = hashlib.md5(f"{source}::{heading}".encode()).hexdigest()
        chunks.append({
            "id": chunk_id, "text": content,
            "source": source, "section": heading,
            "type": "project" if "projects/" in source else "cv"
        })
    return chunks

def embed_and_store(chunks):
    if not chunks:
        return
    texts = [c["text"] for c in chunks]
    vectors = model.encode(texts, convert_to_numpy=True)
    collection.upsert(
        ids        = [c["id"]    for c in chunks],
        embeddings = [v.tolist() for v in vectors],
        documents  = [c["text"]  for c in chunks],
        metadatas  = [{"source": c["source"], "section": c["section"], "type": c["type"]} for c in chunks]
    )

def ingest_file(filepath: str):
    filepath = Path(filepath)
    text   = filepath.read_text(encoding="utf-8")
    source = filepath.name
    chunks = chunk_markdown(text, source)
    embed_and_store(chunks)
    print(f"✓ {filepath.name} — {len(chunks)} chunks")


In [0]:
import os, shutil
# Set your OpenAI API key in .env file or environment
# os.environ["OPENAI_API_KEY"] = "your-key-here"

CHROMA_PATH = "/tmp/portfolio_assistant/chroma_store"
BACKUP_ZIP  = "/Workspace/Users/ariamostajeran99@gmail.com/portfolio-assistant/chroma_backup.zip"

# Restore from backup if chroma sqlite doesn't exist in /tmp
db_file = os.path.join(CHROMA_PATH, "chroma.sqlite3")

if not os.path.exists(db_file):
    if os.path.exists(BACKUP_ZIP):
        os.makedirs(CHROMA_PATH, exist_ok=True)
        shutil.unpack_archive(BACKUP_ZIP, CHROMA_PATH)
        print("✓ Restored from Workspace backup")
    else:
        print("⚠ No backup found — will re-ingest if empty")
else:
    print("✓ Chroma store already on disk")

from sentence_transformers import SentenceTransformer
import chromadb
from openai import OpenAI

model         = SentenceTransformer("BAAI/bge-small-en-v1.5")
client        = OpenAI()
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = chroma_client.get_or_create_collection(
    name="knowledge_base",
    metadata={"hnsw:space": "cosine"}
)
print(f"Connected — {collection.count()} chunks")

# Auto re-ingest + save backup if store is empty
if collection.count() == 0:
    print("Store empty — re-ingesting...")
    BASE = "/Workspace/Users/ariamostajeran99@gmail.com/portfolio-assistant/knowledge"
    ingest_file(f"{BASE}/projects/portfolio_site.md")
    ingest_file(f"{BASE}/cv.md")
    shutil.make_archive(
        "/Workspace/Users/ariamostajeran99@gmail.com/portfolio-assistant/chroma_backup",
        "zip",
        CHROMA_PATH
    )
    print(f"✓ Backup saved — {collection.count()} chunks")

In [0]:
# Each tool is just a function. The agent decides when to call them.

def search_knowledge(query: str, n: int = 3) -> str:
    """
    Search the knowledge base (CV + project docs).
    Returns formatted string of top chunks — ready to inject into a prompt.
    """
    query_vector = model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_vector,
        n_results=n,
        include=["documents", "metadatas", "distances"]
    )

    if not results["documents"][0]:
        return "No relevant information found."

    output = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = round((1 - dist) * 100, 1)
        output.append(
            f"[{meta['source']} / {meta['section']} — {similarity}% match]\n{doc}"
        )

    return "\n\n---\n\n".join(output)


def get_cv_section(section_name: str) -> str:
    """
    Get a specific section from the CV directly by name.
    More precise than semantic search for known section names.
    e.g. section_name = "Skills", "Experience", "Education"
    """
    results = collection.get(
        where={"source": "cv.md"},
        include=["documents", "metadatas"]
    )

    for doc, meta in zip(results["documents"], results["metadatas"]):
        if section_name.lower() in meta["section"].lower():
            return f"[cv.md / {meta['section']}]\n{doc}"

    return f"Section '{section_name}' not found in CV."


# Tool registry — maps tool names to functions and descriptions
TOOLS = {
    "search_knowledge": {
        "fn":          search_knowledge,
        "description": "Search Aria's portfolio knowledge base (CV, projects, skills, experience). Use for any question about Aria.",
        "input":       "A search query string"
    },
    "get_cv_section": {
        "fn":          get_cv_section,
        "description": "Get a specific CV section by name (Skills, Experience, Education, Awards, etc). Use when the question targets a specific CV section.",
        "input":       "Section name as a string"
    }
}

In [0]:
class PortfolioAgent:
    """
    A ReAct agent that answers questions about Aria's portfolio.

    ReAct loop:
    1. LLM decides which tool to call and with what input
    2. Tool is called, result returned
    3. LLM decides if it has enough info or needs another tool call
    4. LLM generates final answer
    5. Answer + exchange added to conversation history
    """

    def __init__(self):
        self.history: List[Dict] = []
        self.system_prompt = self._build_system_prompt()

    def _build_system_prompt(self) -> str:
        tool_descriptions = "\n".join([
            f"- {name}: {info['description']} | Input: {info['input']}"
            for name, info in TOOLS.items()
        ])

        return f"""You are a helpful assistant for Aria Mostajeran's portfolio website.
Aria is an MSc Data Science & AI student at TU/e, finishing September 2025,
actively looking for ML/data science/AI engineering roles.

You have access to these tools to look up information about Aria:
{tool_descriptions}

To use a tool, respond with EXACTLY this format (nothing before or after):
TOOL: tool_name
INPUT: your input here

After seeing the tool result, either:
- Call another tool if you need more information
- Write your final answer if you have enough

Rules:
- ALWAYS use a tool before answering — never answer from memory
- Only state facts that appear in tool results
- If the answer is not in any tool result, say "I don't have that information"
- Be concise and specific — use real names, numbers, technologies
- When referencing projects, mention the tech stack and outcomes"""

    def _call_tool(self, tool_name: str, tool_input: str) -> str:
        if tool_name not in TOOLS:
            return f"Unknown tool: {tool_name}"
        return TOOLS[tool_name]["fn"](tool_input)

    def _parse_tool_call(self, response: str):
        """Extract tool name and input from LLM response"""
        tool_match  = re.search(r'TOOL:\s*(\w+)', response)
        input_match = re.search(r'INPUT:\s*(.+?)(?:\n|$)', response, re.DOTALL)

        if tool_match and input_match:
            return tool_match.group(1).strip(), input_match.group(1).strip()
        return None, None

    def chat(self, user_message: str, verbose: bool = False) -> str:
        # Add user message to history
        self.history.append({"role": "user", "content": user_message})

        messages = [{"role": "system", "content": self.system_prompt}] + self.history

        max_iterations = 3  # prevent infinite tool loops
        iteration = 0

        while iteration < max_iterations:
            iteration += 1

            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=messages,
                temperature=0.2,
                max_tokens=500
            ).choices[0].message.content

            if verbose:
                print(f"\n--- Iteration {iteration} ---")
                print(f"LLM: {response}")

            # Check if LLM wants to call a tool
            tool_name, tool_input = self._parse_tool_call(response)

            if tool_name:
                # Call the tool
                tool_result = self._call_tool(tool_name, tool_input)

                if verbose:
                    print(f"Tool result preview: {tool_result[:200]}...")

                # Add tool call + result to messages for next iteration
                messages.append({"role": "assistant", "content": response})
                messages.append({
                    "role": "user",
                    "content": f"Tool result:\n{tool_result}\n\nNow answer the original question."
                })

            else:
                # No tool call — this is the final answer
                self.history.append({"role": "assistant", "content": response})
                return response

        # Fallback if max iterations hit
        return "I wasn't able to find a complete answer. Please try rephrasing."

    def reset(self):
        """Clear conversation history"""
        self.history = []
        print("Conversation reset")

In [0]:
agent = PortfolioAgent()

questions = [
    "Does Aria have experience with Docker?",
    "What is Aria's strongest ML project?",
    "What makes Aria's portfolio different from a normal CV?",
    "Is Aria available for work?",
    "What is Aria's salary expectation?"   # should refuse
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {agent.chat(q)}")
    print()

In [0]:
# Reset for a fresh conversation
agent.reset()

# This tests whether the agent remembers context across turns
exchanges = [
    "What ML projects has Aria worked on?",
    "Which of those involved time series data?",   # "those" refers to previous answer
    "What model did Aria use for that?",            # "that" refers to time series project
]

for message in exchanges:
    print(f"User: {message}")
    print(f"Agent: {agent.chat(message)}")
    print()

In [0]:
agent.reset()
answer = agent.chat("What ML projects has Aria worked on?", verbose=True)
print(answer)